# IBL Brain-Wide Map Project

Analysis notebook built on the International Brain Laboratory (IBL) Brain-Wide Map dataset.

The dataset contains whole-mouse-brain Neuropixels recordings during a standardized visual decision-making task. This notebook sets up the connection to the public IBL data server, downloads precomputed PSTHs, and defines utility functions for loading and analyzing neural activity.

## Running this notebook

**Google Colab:** Just run all cells. The first cell auto-installs any missing dependencies.

**Locally:** Create an environment and install dependencies first:

```bash
python -m venv .venv
# Windows:  .venv\Scripts\activate
# macOS/Linux:  source .venv/bin/activate
pip install -r requirements.txt
```

Then select the `.venv` as your notebook kernel and run all cells. The setup cell will report any packages still missing.

## Install dependencies, setup, and import libraries

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Colab")
    %pip install -q ONE-api ibllib ibl-style
else:
    print("Running locally — install deps with: pip install -r requirements.txt")

In [ ]:
# When running in jupyter set number of threads to 1
from one.api import ONE
import os
os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')

ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import pandas as pd
from scipy import sparse
import numpy as np
from one.remote.aws import s3_download_file
import zipfile
import tqdm
import scipy.stats as stats
from iblutil.util import Bunch
from scipy.ndimage import gaussian_filter1d

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# This is to view dataframe interactively in Google colab
if 'google.colab' in sys.modules:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()

# This is to style figures (optional — skip gracefully if unavailable)
try:
    from ibl_style.style import figure_style
    figure_style()
except Exception as e:
    print(f"ibl_style not applied ({e}); using default matplotlib style")

## Data Access

### Data Format

This project uses **precomputed PSTHs** (peri-stimulus time histograms) for all *good* clusters in the Brain-Wide Map. The data is provided as three ZIP archives, one per task-event alignment:

| Event | File | Aligned to |
| --- | --- | --- |
| `stimOn` | `data_stimOn.zip` | Stimulus onset |
| `firstMove` | `data_firstMove.zip` | First wheel movement |
| `feedback` | `data_feedback.zip` | Reward / error feedback |

Each archive (~810 MB) contains:

- **`clusters.pqt`** — per-cluster metadata. Key columns: `psth_index` (row into the PSTH array), `uuids` (globally unique cluster id), `cluster_id`, `firing_rate`, `x`/`y`/`z` (CCF coordinates), `atlas_id_allen`, `pid` (probe insertion), `acronym` (Allen CCF brain region), `acronym_allen`.
- **`trials.pqt`** — trial variables: `contrastLeft`, `contrastRight`, `choice`, `probabilityLeft` (block), `feedback`.
- **`sessions.pqt`** — session info mapping `eid` ↔ `pid`, plus `subject`, `lab`.
- **`t.npy`** — PSTH timebase, from −0.5 s to +1.0 s in 10 ms bins (150 bins, `dt = 0.01`).
- **`{pid}.npz`** — one sparse PSTH array per probe insertion, reshaped to `(trials, clusters, nbins)`.

**IDs:** an `eid` identifies an *experimental session*; a `pid` identifies a single *probe insertion*. One session (`eid`) can have multiple probe insertions (`pid`s).

### Data loading utilities

The functions below download, unzip, and load the data into memory. `download_data(event)` fetches and extracts an archive; `load_metadata(event)` reads the metadata tables and PSTH timebase into a `Bunch`; `load_psth(...)` loads a probe insertion's sparse PSTH; and the `get_*_psth_*` helpers aggregate PSTHs by probe insertion, brain region, or specific clusters (optionally split by a task variable).

In [ ]:
def download_data(event):
  assert event in ['firstMove', 'stimOn', 'feedback'], 'event must be one of "firstMove", "stimOn" or "feedback'

  # Dataset name
  fname = f'data_{event}.zip'
  # Remote location of data
  s3_data_path = f'sample_data/Neuromatch/{fname}'
  # Local location to download data to
  save_path = one.cache_dir.joinpath('Neuromatch', fname)
  save_path.parent.mkdir(exist_ok=True, parents=True)

  # Download file
  file = s3_download_file(s3_data_path, save_path)
  # Unzip content
  with zipfile.ZipFile(file, 'r') as zip_ref:
    zip_ref.extractall(save_path.parent)


def get_data_path(event):

  return one.cache_dir.joinpath('Neuromatch', f'data_{event}')

In [ ]:
def load_metadata(event):
  metadata = Bunch()
  data_path = get_data_path(event)
  metadata['clusters'] = pd.read_parquet(data_path.joinpath('clusters.pqt'))
  metadata['trials'] = pd.read_parquet(data_path.joinpath('trials.pqt'))
  metadata['sessions'] = pd.read_parquet(data_path.joinpath('sessions.pqt'))
  metadata['times'] = np.load(data_path.joinpath('t.npy'))
  metadata['nbins'] = metadata['times'].size
  metadata['dt'] = np.round(np.median(np.diff(metadata['times'])), 2)
  metadata['data_path'] = data_path

  return metadata


def load_times(data_path):
  return np.load(data_path.joinpath('t.npy'))


def load_psth(data_path, pid, nbins=150):
    psth = sparse.load_npz(data_path.joinpath(f'{pid}.npz')).toarray()
    psth = psth.reshape(psth.shape[0], -1, nbins)
    return psth

In [ ]:
def split_trials_by_variable(trials, split='contrast'):
  trials = trials.set_index('psth_index')
  if split == 'contrast':
    trials['contrast'] = np.nansum([trials['contrastLeft'], trials['contrastRight']], axis=0) * 100
    grp = trials.groupby('contrast')
  elif split == 'signed contrast':
    trials['signedContrast'] = np.nansum([-1 * trials['contrastLeft'], trials['contrastRight']], axis=0) * 100
    grp = trials.groupby('signedContrast')
  elif split == 'stimulus':
    trials['stimulus'] = 'right'
    trials.loc[trials['contrastRight'].isna(), 'stimulus'] = 'left'
    grp = trials.groupby('stimulus')
  elif split == 'choice':
    grp = trials.groupby('choice')
  elif split == 'block':
    grp = trials.groupby('probabilityLeft')
  else:
    raise NotImplementedError('split must be one of "contrast", "signed contrast", "stimulus", "choice" or "block"')

  return grp.groups


def get_avg_psth_for_insertion(pid, meta, reg=None, uuids=None, split=None):

  df = meta.clusters[meta.clusters['pid'] == pid]
  df = df[['acronym', 'pid', 'uuids', 'cluster_id', 'psth_index']]
  sp = load_psth(meta.data_path, pid, nbins=meta.nbins)

  if reg is not None:
    in_reg = df['acronym'] == reg
    sp = sp[:, in_reg.values, :]
    df = df[in_reg].reset_index(drop=True)

  if uuids is not None:
    in_uuid = df['uuids'].isin(uuids)
    sp = sp[:, in_uuid.values, :]
    df = df[in_uuid].reset_index(drop=True)

  if split is None:
    psth = sp.mean(axis=0) / meta['dt']
  else:
    psth = dict()
    eid = meta.sessions[meta.sessions['pid'] == pid].iloc[0]['eid']
    trials = meta.trials[meta.trials['eid'] == eid].reset_index(drop=True)
    grps = split_trials_by_variable(trials, split=split)

    for key, vals in grps.items():
      psth[key] = sp[vals, :, :].mean(axis=0)

  return psth, df

def get_avg_psth_for_region(reg, meta, split=None):
  clusters = meta.clusters[meta.clusters['acronym'] == reg]
  pids = clusters['pid'].unique()
  all_df = []
  all_psth = []
  for pid in pids:
    psth, df = get_avg_psth_for_insertion(pid, meta, reg=reg, split=split)
    all_df.append(df)
    all_psth.append(psth)

  all_df = pd.concat(all_df).reset_index(drop=True)
  if split is None:
    all_psth = np.concatenate(all_psth)
  else:
    all_psth = {key: np.concatenate([d[key] for d in all_psth if key in d.keys()])
    for key in all_psth[0]}


  return all_psth, all_df


def get_avg_psth_for_clusters(uuids, meta, split=None):
  clusters = meta.clusters[meta.clusters['uuids'].isin(uuids)]
  pids = clusters['pid'].unique()
  all_df = []
  all_psth = []
  for pid in pids:
    psth, df = get_avg_psth_for_insertion(pid, meta, uuids=uuids, split=split)
    all_df.append(df)
    all_psth.append(psth)

  all_df = pd.concat(all_df).reset_index(drop=True)
  if split is None:
    all_psth = np.concatenate(all_psth)
  else:
    all_psth = {key: np.concatenate([d[key] for d in all_psth if key in d.keys()])
    for key in all_psth[0]}

  return all_psth, all_df



def get_psth_for_insertion(pid, meta, reg=None, uuids=None):

  df = meta.clusters[meta.clusters['pid'] == pid]
  df = df[['acronym', 'pid', 'uuids', 'cluster_id', 'psth_index']]
  sp = load_psth(meta.data_path, pid, nbins=meta.nbins)

  if reg is not None:
    in_reg = df['acronym'] == reg
    sp = sp[:, in_reg.values, :]
    df = df[in_reg].reset_index(drop=True)

  if uuids is not None:
    in_uuid = df['uuids'].isin(uuids)
    sp = sp[:, in_uuid.values, :]
    df = df[in_uuid].reset_index(drop=True)


  eid = meta.sessions[meta.sessions['pid'] == pid].iloc[0]['eid']
  trials = meta.trials[meta.trials['eid'] == eid].reset_index(drop=True)
  psth = sp / meta['dt']

  return psth, df, trials


def get_psth_for_region(reg, meta):

  clusters = meta.clusters[meta.clusters['acronym'] == reg]
  pids = clusters['pid'].unique()
  all_clust = []
  all_psth = []
  all_trials = []
  for pid in pids:
    psth, clust, trials = get_psth_for_insertion(pid, meta, reg=reg)
    all_clust.append(clust)
    all_psth.append(psth)
    all_trials.append(trials)

  return all_psth, all_clust, all_trials


def get_psth_for_clusters(uuids, meta):

  clusters = meta.clusters[meta.clusters['uuids'].isin(uuids)]
  pids = clusters['pid'].unique()
  all_clust = []
  all_psth = []
  all_trials = []
  for pid in pids:
    psth, clust, trials = get_psth_for_insertion(pid, meta, uuids=uuids)
    all_clust.append(clust)
    all_psth.append(psth)
    all_trials.append(trials)

  return all_psth, all_clust, all_trials

### Explore the data

Pick an event alignment, download the data, and inspect the metadata tables. Change `event` to `'firstMove'` or `'feedback'` to align to a different task event.

In [ ]:
event = 'stimOn'
download_data(event)
meta = load_metadata(event)

print('metadata keys:', list(meta.keys()))
print('n clusters:', len(meta.clusters))
print('n bins:', meta.nbins, '| dt:', meta.dt, 's')
print('time range:', meta.times[0], 'to', meta.times[-1], 's')

**Clusters** — each row is a single neuron. The `acronym` column gives the Allen CCF brain region; `pid` is the probe insertion; `psth_index` maps the cluster to its row in the PSTH arrays; `cluster_id` and `uuids` identify the neuron.

In [ ]:
meta.clusters.head(100)

**Available brain regions** — use these `acronym` values with `get_avg_psth_for_region(reg, meta, ...)`. The counts show how many good clusters were recorded in each region.

In [ ]:
meta.clusters['acronym'].value_counts().head(30)

**Trials** — task variables per trial: stimulus contrast on each side (`contrastLeft`/`contrastRight`), the animal's `choice`, the block prior (`probabilityLeft`), and the `feedback` (reward/error).

In [ ]:
meta.trials.head()

## Analysis

This is your scaffold — build your own project below. The utilities above give you PSTHs aligned to `stimOn`, `firstMove`, or `feedback`, filterable by brain region or cluster, and splittable by task variable.

**Example: average PSTH for a brain region, split by a task variable**

```python
reg = 'VISp'  # pick a region from meta.clusters['acronym'].value_counts()
psth, df = get_avg_psth_for_region(reg, meta, split='choice')

for key, arr in psth.items():
    plt.plot(meta.times, arr.mean(axis=0), label=f'{key}')
plt.axvline(0, color='k', ls='--', lw=1)
plt.xlabel(f'Time from {event} (s)')
plt.ylabel('Firing rate (sp/s)')
plt.legend(title='choice')
plt.show()
```

**Ideas to explore:**

- Compare neural responses across brain regions or task events
- Split PSTHs by `contrast`, `signed contrast`, `stimulus`, `choice`, or `block`
- Run PCA on population activity (`PCA` is imported)
- Decode stimulus or choice from population activity (`LogisticRegression` + `train_test_split` are imported)
- Smooth PSTHs with `gaussian_filter1d` for cleaner traces

## Q3 — Decoding choice from pre-movement population activity

**Question:** Can we predict the animal's upcoming left/right `choice` from population activity *before* it moves, and which brain regions carry the most choice information?

**Design decisions (locked):**

- **Event alignment:** `firstMove`. Time `t = 0` is first movement onset.
- **Feature window:** `-0.1 s < t < 0 s` (the 100 ms *before* movement). Using only pre-movement bins avoids the motor confound — we decode the upcoming decision, not the movement itself.
- **Features:** per neuron, mean firing rate across the window → one feature per unit, giving a `(trials × units)` matrix per session.
- **Labels:** `trials.choice` (IBL codes right/left as `+1` / `-1`). No-go trials (`choice == 0`) are dropped.
- **Classifier:** `LogisticRegression(random_state=0, max_iter=1000)`, matching the tutorial.
- **Validation:** 5-fold **StratifiedKFold** cross-validation (keeps class balance per fold), reported as mean CV accuracy.
- **Baseline:** a **shuffled-label** null — the same CV pipeline run on permuted choice labels — gives an empirical chance level per session-region (robust to class imbalance, unlike a fixed 0.5).
- **Cross-session pooling:** each brain region is recorded across many insertions in *different* sessions with non-corresponding trials. We therefore decode **each (region, session) independently** and report the **mean CV accuracy across sessions** for that region, with spread. No cross-session trial alignment is assumed.
- **Inclusion gates:** a session-region must have at least `MIN_UNITS` simultaneously-recorded units and `MIN_TRIALS` usable trials (after dropping no-go) to be decoded.

**Deliverable:** regions ranked by mean pre-movement choice-decoding accuracy, each compared against its shuffled-label baseline.

In [ ]:
# Load the firstMovement-aligned dataset for Q3.
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

download_data('firstMove')
meta_fm = load_metadata('firstMove')

# Pre-movement feature window: the 100 ms immediately before movement onset.
# Using only t < 0 avoids the motor confound (we never see the movement itself).
PRE_WINDOW = (-0.1, 0.0)
time_mask_fm = np.bitwise_and(meta_fm.times < PRE_WINDOW[1], meta_fm.times >= PRE_WINDOW[0])

print('firstMove metadata loaded')
print('n clusters:', len(meta_fm.clusters))
print('time range:', meta_fm.times[0], 'to', meta_fm.times[-1], 's | dt:', meta_fm.dt)
print(f'feature window {PRE_WINDOW} → {int(time_mask_fm.sum())} bins')

In [ ]:
# Decode choice for a single (region, session) feature matrix.
# X: (trials, units) pre-movement mean firing rates. y: choice labels (+1/-1).
# Returns mean 5-fold StratifiedKFold CV accuracy plus a shuffled-label null.
N_SPLITS = 5          # CV folds
N_SHUFFLES = 20       # permutations for the empirical chance baseline

def decode_region_session(X, y, n_splits=N_SPLITS, n_shuffles=N_SHUFFLES, seed=0):
    """Return (cv_acc, shuffle_mean, shuffle_std) for one session-region.

    cv_acc      : mean StratifiedKFold CV accuracy on true labels.
    shuffle_mean: mean CV accuracy over n_shuffles label permutations (empirical chance).
    shuffle_std : std of the shuffled-label CV accuracies.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    # Standardize features INSIDE each CV fold (fit on train only) to avoid
    # leakage. Scaling also fixes the lbfgs ConvergenceWarning caused by
    # firing-rate features spanning very different magnitudes.
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(random_state=seed, max_iter=1000),
    )
    cv_acc = cross_val_score(clf, X, y, cv=cv, scoring='accuracy').mean()

    rng = np.random.default_rng(seed)
    shuffle_accs = []
    for _ in range(n_shuffles):
        y_perm = rng.permutation(y)
        cv_s = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        acc = cross_val_score(clf, X, y_perm, cv=cv_s, scoring='accuracy').mean()
        shuffle_accs.append(acc)
    shuffle_accs = np.array(shuffle_accs)
    return cv_acc, shuffle_accs.mean(), shuffle_accs.std()

In [ ]:
# Decode choice for a region across all sessions it appears in, then average.
MIN_UNITS = 5    # minimum simultaneously-recorded units per session-region
MIN_TRIALS = 20  # minimum usable (non no-go) trials per session-region

def decode_choice_for_region(reg, meta, time_mask):
    """Decode upcoming choice per session for `reg`; return a per-session results list.

    Each session is decoded independently (trials do not correspond across sessions).
    A session-region is skipped unless it clears MIN_UNITS and MIN_TRIALS and has
    both choice classes present.
    """
    all_psth, all_clust, all_trials = get_psth_for_region(reg, meta)
    sessions = []
    for psth, clust, trials in zip(all_psth, all_clust, all_trials):
        # psth: (trials, units, nbins). Pre-movement mean firing rate per unit.
        X = psth[:, :, time_mask].mean(axis=2)
        y = trials['choice'].values
        # Drop no-go trials (choice == 0); keep only left/right decisions.
        keep = y != 0
        X, y = X[keep], y[keep]
        # Inclusion gates.
        n_trials, n_units = X.shape
        if n_units < MIN_UNITS or n_trials < MIN_TRIALS:
            continue
        if np.unique(y).size < 2:
            continue
        # Need each class to have at least N_SPLITS members for StratifiedKFold.
        _, counts = np.unique(y, return_counts=True)
        if counts.min() < N_SPLITS:
            continue
        cv_acc, shuf_mean, shuf_std = decode_region_session(X, y)
        sessions.append({
            'region': reg,
            'n_units': int(n_units),
            'n_trials': int(n_trials),
            'cv_acc': cv_acc,
            'shuffle_acc': shuf_mean,
            'shuffle_std': shuf_std,
        })
    return sessions

In [ ]:
# Run the decoder across every region, skipping non-brain labels.
regions = [r for r in meta_fm.clusters['acronym'].unique() if r not in ('root', 'void')]
print(f'decoding {len(regions)} regions...')

session_records = []
for reg in tqdm.tqdm(regions):
    session_records.extend(decode_choice_for_region(reg, meta_fm, time_mask_fm))

session_df = pd.DataFrame(session_records)

# Aggregate to one row per region: mean CV accuracy across its sessions.
region_df = (
    session_df
    .groupby('region')
    .agg(
        n_sessions=('cv_acc', 'size'),
        mean_cv_acc=('cv_acc', 'mean'),
        std_cv_acc=('cv_acc', 'std'),
        mean_shuffle=('shuffle_acc', 'mean'),
        total_units=('n_units', 'sum'),
    )
    .reset_index()
)
# Rank by how far above its own shuffle baseline each region decodes.
region_df['above_chance'] = region_df['mean_cv_acc'] - region_df['mean_shuffle']
region_df = region_df.sort_values('mean_cv_acc', ascending=False).reset_index(drop=True)
region_df.head(25)

In [ ]:
# Plot the top regions by mean pre-movement choice-decoding accuracy vs shuffle baseline.
TOP_N = 20
plot_df = region_df.head(TOP_N).iloc[::-1]  # reverse so best is at the top of a horizontal bar

fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(plot_df))))
ypos = np.arange(len(plot_df))
ax.barh(ypos, plot_df['mean_cv_acc'], xerr=plot_df['std_cv_acc'].fillna(0),
        color='steelblue', alpha=0.85, label='CV accuracy')
ax.scatter(plot_df['mean_shuffle'], ypos, color='crimson', zorder=3,
           marker='|', s=200, label='shuffled-label baseline')
ax.set_yticks(ypos)
ax.set_yticklabels(plot_df['region'])
ax.set_xlabel('choice-decoding accuracy (pre-movement)')
ax.set_title(f'Top {TOP_N} regions — choice decoding vs shuffle baseline')
ax.legend(loc='lower right')
sns.despine()
plt.tight_layout()
plt.show()